https://www.cnblogs.com/tian777/p/15126018.html

tqdm 是一个快速、可扩展的 Python 进度条库，可以在长循环中添加一个进度提示信息，用户只需要封装任意的迭代器 tqdm(iterator)。


In [1]:
import asyncio
# 协程是一种特殊的函数，使用 async def 定义，可以被暂停和恢复执行。协程通过 await 关键字调用其他协程或异步操作
async def eval(tasks):
    start = time.time()
    # 使用 tqdm.tqdm 创建一个进度条。
    # total=len(tasks) 表示进度条的总任务数等于任务列表的长度
    with tqdm.tqdm(total=len(tasks)) as pbar:
        # 定义了一个内部的异步函数 _tsk，它接收一个协程 coro 作为参数
        # coro 是任务的协程对象。
        # pbar.update(1) 表示进度条增加1。
        async def _tsk(coro): 
            # 使用 await coro 等待协程完成，并获取其返回值 ret
            ret = await coro
            pbar.update(1)
            return ret
        tasks = [_tsk(t) for t in tasks]
        responses = await asyncio.gather(*tasks)
    end = time.time()
    print(f"Time taken: {end - start} seconds")
    return responses

# 使用 asyncio.run() 来运行主事件循环
ret = asyncio.run(eval(tasks))

NameError: name 'tasks' is not defined

### process_map 是 tqdm 库的一个高级功能，用于在多进程环境中显示进度条。

#### process_map 的工作原理如下：

1、创建进程池：
- 使用 concurrent.futures.ProcessPoolExecutor 创建一个进程池，指定最大工作进程数 max_workers。

2、分块处理数据：
- 将可迭代对象 iterable 分成多个块，每个块的大小由 chunksize 决定。
- 每个工作进程处理一个块的数据。

3、显示进度条：
- 使用 tqdm 显示进度条，实时更新任务的完成情况。

4、收集结果：
- 将所有工作进程的返回值收集起来，返回一个结果列表。

In [2]:
from tqdm import tqdm
import time

for i in tqdm(range(100)):
    time.sleep(0.01)  # 模拟耗时操作

100%|█████████████████████████████████████████| 100/100 [00:01<00:00, 60.30it/s]


In [8]:
import logging
from tqdm.contrib.concurrent import process_map
import time

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def worker(x):
    try:
        logging.info(f"Processing {x}")
        time.sleep(0.1)  # 模拟耗时操作
        if x == 5:
            raise ValueError("Something went wrong with x=5")
        return x * x
    except Exception as e:
        logging.error(f"Error in worker: {e}")
        return None

if __name__ == "__main__":
    data = range(10)
    results = process_map(worker, data, max_workers=4, chunksize=1, desc="Processing")
    print(results)

Processing:   0%|          | 0/10 [00:00<?, ?it/s]

Process SpawnProcess-23:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/gym3.10.10/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/gym3.10.10/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/envs/gym3.10.10/lib/python3.10/concurrent/futures/process.py", line 240, in _process_worker
    call_item = call_queue.get(block=True)
  File "/opt/anaconda3/envs/gym3.10.10/lib/python3.10/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'worker' on <module '__main__' (built-in)>
Process SpawnProcess-21:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/gym3.10.10/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/gym3.10.10/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.